# 패키지 import

# Phase 1 

  ✓ pyaedt_actuator 패키지 구조 생성
  ✓ ActuatorDesignSpec 클래스 완성

# Phase 2 (이번주 )

  □ geometry.py 함수 구현 (step-by-step)
  □ 2D 기하학 생성 및 시각화 검증

# Phase 3 Maxwell 연동

  □ AEDT 프로젝트 생성 및 설계 파라미터 적용
  □ 시뮬레이션 실행 및 결과 추출

In [ ]:
✅ Day 6-7 (다음 주 초)
  □ maxwell_bridge.py 개발
  □ Maxwell 모델 생성 검증

## Maxwell GUI에서 Actuator 모델링 (라인바이라인)

이 섹션은 Maxwell GUI에서 수행하는 모든 작업을 **pyAEDT 명령어로 라인바이라인 구현**합니다.

## 📌 목표
회전 액추에이터(Rotary Motor)를 Maxwell 2D에서 모델링하는 전체 과정을 단계별로 실행

## 🎯 워크플로우
```
Step 1: 프로젝트 생성 & 기본 설정
Step 2: 기하학 생성 (스테이터, 로터, 에어갭)
Step 3: 재료 할당 & 코일 정의
Step 4: 경계 & 여기 조건 설정
Step 5: 메시 생성
Step 6: 시뮬레이션 실행 & 결과 추출
```

## Step 1: 라이브러리 Import & 기본 설정

In [ ]:
Notebook_path='E:\KDH\gitDosa_Actuator\SimMaxwell'

In [23]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# pyAEDT import
from ansys.aedt.core import Maxwell2d
from ansys.aedt.core import Desktop

# 설정
AEDT_VERSION = "2026.1"
NON_GRAPHICAL = False  # GUI 표시 (False = 표시)

print("✅ Libraries imported successfully")
print(f"   AEDT Version: {AEDT_VERSION}")
print(f"   Non-graphical mode: {NON_GRAPHICAL}")

✅ Libraries imported successfully
   AEDT Version: 2026.1
   Non-graphical mode: False


## Step 2: 설계 파라미터 정의

In [24]:
# 기하학 및 성능 파라미터
PARAMS = {
    # 스테이터
    'stator_outer_d': 60,      # 외경 (mm)
    'stator_inner_d': 40,      # 내경 (mm)
    'stator_length': 50,       # 길이 (mm)
    
    # 로터
    'rotor_outer_d': 39,       # 외경 (mm)
    'rotor_inner_d': 20,       # 내경 (mm)
    
    # 에어갭
    'air_gap': 0.5,            # 간격 (mm)
    
    # 코일
    'num_coils': 6,            # 코일 개수
    'turns_per_coil': 100,     # 코일당 권선 수
    'wire_diameter': 0.5,      # 도선 직경 (mm)
    
    # 극
    'pole_pairs': 4,           # 극 쌍수 (스테이터 슬롯 = pole_pairs * 3)
    
    # 재료
    'stator_material': "M19",  # 전자석강
    'rotor_material': "M36",
    'wire_material': "Copper",
}

# 파라미터 출력
print("📊 Actuator Design Parameters")
print("=" * 50)
print(f"{'Parameter':<25} {'Value':>20}")
print("=" * 50)
for key, value in sorted(PARAMS.items()):
    print(f"{key:<25} {str(value):>20}")

# 계산된 값
num_slots = PARAMS['pole_pairs'] * 3
total_turns = PARAMS['num_coils'] * PARAMS['turns_per_coil']

print("=" * 50)
print(f"{'Calculated Values':<25}")
print("=" * 50)
print(f"  Number of slots: {num_slots} (pole_pairs={PARAMS['pole_pairs']} × 3)")
print(f"  Total turns: {total_turns} (coils={PARAMS['num_coils']} × turns={PARAMS['turns_per_coil']})")

📊 Actuator Design Parameters
Parameter                                Value
air_gap                                    0.5
num_coils                                    6
pole_pairs                                   4
rotor_inner_d                               20
rotor_material                             M36
rotor_outer_d                               39
stator_inner_d                              40
stator_length                               50
stator_material                            M19
stator_outer_d                              60
turns_per_coil                             100
wire_diameter                              0.5
wire_material                           Copper
Calculated Values        
  Number of slots: 12 (pole_pairs=4 × 3)
  Total turns: 600 (coils=6 × turns=100)


## Step 3: Maxwell 프로젝트 생성 및 초기화

In [25]:
# PROJECT_NAME = f"RotaryMotor_{PARAMS['pole_pairs']}P_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
PROJECT_NAME = "Axi_Magnetostatic_Actuator"
SOLUTION_TYPE = "MagnetostaticZ"
Design_NAME = "01_Actuator" 
import os
fullfilePath=os.path.join(Notebook_path, f"{PROJECT_NAME}.aedt")
print(f"\n✅ Project file path: {fullfilePath}")
print("🚀 Creating Maxwell 2D project...")
print(f"   Project: {PROJECT_NAME}")
print(f"   Solution: {SOLUTION_TYPE}")



✅ Project file path: E:\KDH\gitDosa_Actuator\SimMaxwell\Axi_Magnetostatic_Actuator.aedt
🚀 Creating Maxwell 2D project...
   Project: Axi_Magnetostatic_Actuator
   Solution: MagnetostaticZ


In [ ]:

# Maxwell 2D 인스턴스 생성
m2d = Maxwell2d(
    project=PROJECT_NAME,
    design=Design_NAME,
    version=AEDT_VERSION,
    non_graphical=NON_GRAPHICAL,
    new_desktop=False,
)

print(f"\n✅ Project created successfully")
print(f"   Project name: {m2d.project_name}")
print(f"   Design name: {m2d.design_name}")
print(f"   Solution type: {m2d.solution_type}")
print(f"   Model units: {m2d.modeler.model_units}")
print(f"   Working directory: {m2d.working_directory}")

In [ ]:
m2d.modeler.model_units = "mm"  # Global units used in geometry creation

In [ ]:
m2d.save_project(file_name=fullfilePath)

## Step 4: 기하학 생성 - TRC(교육자료기준)

In [ ]:
# Desktop 인스턴스 생성 후 프로젝트 로드
desktop = Desktop(version=AEDT_VERSION, non_graphical=NON_GRAPHICAL)
m2d = desktop.load_project(fullfilePath)

PyAEDT INFO: Python version 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)].
PyAEDT INFO: PyAEDT version 0.25.0.
PyAEDT INFO: Initializing new Desktop session.
PyAEDT INFO: AEDT version 2026.1.
PyAEDT INFO: New AEDT session is starting on gRPC port 62553.
PyAEDT INFO: Starting new AEDT gRPC session on port 62553.
PyAEDT INFO: Launching AEDT server with gRPC transport mode: wnua
PyAEDT INFO: Electronics Desktop started on gRPC port 62553 after 27.0 seconds.
PyAEDT INFO: AEDT installation Path C:\Program Files\ANSYS Inc\v261\AnsysEM
PyAEDT INFO: Connected to AEDT gRPC session on port 62553.


In [ ]:
# design parameters
m2d["move"] = "0mm"  # Displacement applied to anchor

# Coil
coil_m = m2d.modeler.create_rectangle(origin=["3mm", "0mm", "7mm"], sizes=[-14, 6], name="Coil", material="Copper")
# Anchor
anchor_m = m2d.modeler.create_rectangle(
    origin=["0mm", "0mm", "13mm - move"],
    sizes=[-8, 2],
    name="Anchor",
    material="steel_1008",
)

# Housing
points_housing = [
    [0, 0, 0],
    [0, 0, -10],
    [12, 0, -10],
    [12, 0, 10],
    [2.5, 0, 10],
    [2.5, 0, 8],
    [10, 0, 8],
    [10, 0, -8],
    [2, 0, -8],
    [2, 0, 0],
]

# Create 
housing_m = m2d.modeler.create_polyline(points_housing, close_surface=True, name="Housing", material="steel_1008")
m2d.modeler.cover_lines(housing_m)

In [ ]:

# # Maxwell에 스테이터 생성
# stator = m2d.modeler.create_polyline(stator_pts, close_line=True)
# stator.name = "Stator"
# stator.material_name = PARAMS['stator_material']

# print(f"✅ Stator created")
# print(f"   Name: {stator.name}")
# print(f"   Material: {stator.material_name}")
# print(f"   Vertices: {len(stator_pts)}")
# print(f"   Outer diameter: {PARAMS['stator_outer_d']} mm")
# print(f"   Inner diameter: {PARAMS['stator_inner_d']} mm")

### Step 5: 기하학 생성 - 로터

In [ ]:
print("\n🔧 Creating Rotor...")

# 로터 좌표
r_rotor = PARAMS['rotor_outer_d'] / 2
r_rotor_inner = PARAMS['rotor_inner_d'] / 2

rotor_outer_x = r_rotor * np.cos(theta)
rotor_outer_y = r_rotor * np.sin(theta)

rotor_inner_x = r_rotor_inner * np.cos(theta)
rotor_inner_y = r_rotor_inner * np.sin(theta)

# 로터 경계 점
rotor_pts = []
for i in range(len(theta)):
    rotor_pts.append([rotor_outer_x[i], rotor_outer_y[i]])
for i in range(len(theta)-1, -1, -1):
    rotor_pts.append([rotor_inner_x[i], rotor_inner_y[i]])

# # Maxwell에 로터 생성
# rotor = m2d.modeler.create_polyline(rotor_pts, close_line=True)
# rotor.name = "Rotor"
# rotor.material_name = PARAMS['rotor_material']

# print(f"✅ Rotor created")
# print(f"   Name: {rotor.name}")
# print(f"   Material: {rotor.material_name}")
# print(f"   Vertices: {len(rotor_pts)}")
# print(f"   Outer diameter: {PARAMS['rotor_outer_d']} mm")
# print(f"   Inner diameter: {PARAMS['rotor_inner_d']} mm")

### Step 6: 기하학 생성 - 에어갭

In [ ]:
print("\n🔧 Creating Air Gap...")

# 에어갭 좌표 (로터 외경과 스테이터 내경 사이)
r_airgap_outer = PARAMS['rotor_outer_d'] / 2 + PARAMS['air_gap']
r_airgap_inner = PARAMS['rotor_outer_d'] / 2

airgap_outer_x = r_airgap_outer * np.cos(theta)
airgap_outer_y = r_airgap_outer * np.sin(theta)

airgap_inner_x = r_airgap_inner * np.cos(theta)
airgap_inner_y = r_airgap_inner * np.sin(theta)

# 에어갭 경계 점
airgap_pts = []
for i in range(len(theta)):
    airgap_pts.append([airgap_outer_x[i], airgap_outer_y[i]])
for i in range(len(theta)-1, -1, -1):
    airgap_pts.append([airgap_inner_x[i], airgap_inner_y[i]])

# # Maxwell에 에어갭 생성
# airgap = m2d.modeler.create_polyline(airgap_pts, close_line=True)
# airgap.name = "AirGap"
# airgap.material_name = "Air"

# print(f"✅ Air Gap created")
# print(f"   Name: {airgap.name}")
# print(f"   Material: {airgap.material_name}")
# print(f"   Gap size: {PARAMS['air_gap']} mm")
# print(f"   Inner radius: {r_airgap_inner} mm")
# print(f"   Outer radius: {r_airgap_outer} mm")

### Step 7: 기하학 시각화 및 검증

In [ ]:
print("\n📊 Visualizing Geometry...")

fig, ax = plt.subplots(1, 1, figsize=(12, 12))

# 스테이터
ax.plot(stator_outer_x, stator_outer_y, 'b-', linewidth=2.5, label='Stator Outer')
ax.plot(stator_inner_x, stator_inner_y, 'b--', linewidth=1.5, alpha=0.7, label='Stator Inner (Slots)')
ax.fill_between(stator_outer_x, stator_outer_y, alpha=0.1, color='blue')

# 로터
ax.plot(rotor_outer_x, rotor_outer_y, 'r-', linewidth=2.5, label='Rotor Outer')
ax.plot(rotor_inner_x, rotor_inner_y, 'r--', linewidth=1.5, alpha=0.7, label='Rotor Inner (Shaft)')
ax.fill_between(rotor_outer_x, rotor_outer_y, alpha=0.1, color='red')

# 에어갭
ax.plot(airgap_outer_x, airgap_outer_y, 'g-', linewidth=1.5, alpha=0.8, label='Air Gap Outer')
ax.plot(airgap_inner_x, airgap_inner_y, 'g--', linewidth=1, alpha=0.5)

# 슬롯 표시 (선택)
num_slots = PARAMS['pole_pairs'] * 3
slot_angle = 360 / num_slots
for slot_id in range(num_slots):
    angle = np.deg2rad(slot_id * slot_angle)
    slot_r = (PARAMS['stator_inner_d'] / 2) - 2
    slot_x = slot_r * np.cos(angle)
    slot_y = slot_r * np.sin(angle)
    ax.plot(slot_x, slot_y, 'bs', markersize=6, alpha=0.5)

# 레이아웃
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)
ax.legend(fontsize=10, loc='upper right')
ax.set_title(f"Actuator Geometry: {PARAMS['pole_pairs']} Pole Pairs ({num_slots} Slots)", fontsize=14, fontweight='bold')
ax.set_xlabel("X (mm)", fontsize=11)
ax.set_ylabel("Y (mm)", fontsize=11)

# 치수 표시
mid_angle = np.deg2rad(0)
ax.annotate('', xy=(r_outer*np.cos(mid_angle), 0), xytext=(0, 0),
            arrowprops=dict(arrowstyle='<->', color='black', lw=1.5))
ax.text(r_outer/2, -3, f"D={PARAMS['stator_outer_d']}mm", fontsize=9, ha='center')

plt.tight_layout()
plt.show()

print(f"✅ Geometry visualization completed")
print(f"   Stator: {PARAMS['stator_outer_d']}∅ × {PARAMS['stator_inner_d']}∅")
print(f"   Rotor: {PARAMS['rotor_outer_d']}∅ × {PARAMS['rotor_inner_d']}∅")
print(f"   Air Gap: {PARAMS['air_gap']} mm")
print(f"   Slots: {num_slots} (pole_pairs × 3)")

### Step 8: 코일 영역 생성 및 와인딩 할당

In [ ]:
print("\n⚡ Creating Coil Regions...")

num_slots = PARAMS['pole_pairs'] * 3
slot_angle = 360 / num_slots
coil_regions = []

# for slot_id in range(num_slots):
#     angle_deg = slot_id * slot_angle
#     angle_rad = np.deg2rad(angle_deg)
    
#     # 슬롯 중심 위치 (스테이터 내경 근처)
#     slot_r = (PARAMS['stator_inner_d'] / 2) - 2  # 슬롯 깊이
#     slot_center_x = slot_r * np.cos(angle_rad)
#     slot_center_y = slot_r * np.sin(angle_rad)
    
#     # 슬롯 영역 생성 (작은 원형 영역)
#     slot_radius = 2
#     coil_theta = np.linspace(0, 2*np.pi, 20)
#     coil_x = slot_center_x + slot_radius * np.cos(coil_theta)
#     coil_y = slot_center_y + slot_radius * np.sin(coil_theta)
    
#     coil_pts = [[coil_x[i], coil_y[i]] for i in range(len(coil_theta))]
    
    # # Maxwell에 코일 영역 생성
    # try:
    #     coil = m2d.modeler.create_polyline(coil_pts, close_line=True)
    #     coil.name = f"Coil_{slot_id:02d}"
    #     coil.material_name = "Copper"
    #     coil_regions.append(coil)
    # except Exception as e:
    #     print(f"  ⚠️ Warning creating coil {slot_id}: {e}")

print(f"✅ Created {len(coil_regions)} coil regions")

# 3상 권선 그룹 정의
phase_A_ids = list(range(0, num_slots, 3))
phase_B_ids = list(range(1, num_slots, 3))
phase_C_ids = list(range(2, num_slots, 3))

phase_coils = {
    'Phase_A': [coil_regions[i] for i in phase_A_ids],
    'Phase_B': [coil_regions[i] for i in phase_B_ids],
    'Phase_C': [coil_regions[i] for i in phase_C_ids],
}

# # 각 상별 와인딩 할당
# for phase_name, coils in phase_coils.items():
#     coil_names = [c.name for c in coils]
    
#     try:
#         m2d.assign_coil(
#             coil_names=coil_names,
#             name=phase_name,
#             type="Solid"
#         )
#         print(f"✅ Winding assigned: {phase_name} ({len(coils)} coils × {PARAMS['turns_per_coil']} turns)")
#     except Exception as e:
#         print(f"  ⚠️ Warning assigning winding {phase_name}: {e}")

## Step 9: 경계 조건 설정

In [ ]:
region_m = m2d.modeler.create_region(pad_percent=100)
region_m.material_name = "vacuum"


In [ ]:
m2d.assign_vector_potential(assignment=region_m.edges, boundary="VectorPotential1")


In [ ]:
print("\n🔗 Setting Boundary Conditions...")

# 1. 외부 경계 (Dirichlet A=0) - 스테이터 외부
try:
    m2d.assign_dirichlet_boundary(
        object_list=[stator],
        name="BC_Dirichlet_A0"
    )
    print(f"✅ Dirichlet boundary set (A=0 on stator outer)")
except Exception as e:
    print(f"  ⚠️ Could not set Dirichlet boundary: {e}")

# 2. 대칭 경계 (Y축 대칭) - 선택
try:
    # m2d.assign_symmetry(axes="Y", name="BC_Symmetry_Y")
    # print(f"✅ Symmetry boundary set (Y-axis)")
    print(f"⚠️ Symmetry boundary skipped (optional)")
except Exception as e:
    print(f"  ⚠️ Could not set symmetry: {e}")

# 3. 회전 대역 (Rotating Band) - 로터 회전 고려
try:
    # Rotating band를 로터 위치에 설정
    # m2d.assign_rotating_band(
    #     axis="Z",
    #     origin=[0, 0, 0],
    #     outer_radius=PARAMS['rotor_outer_d']/2 + 2,
    #     inner_radius=0,
    #     name="RotatingBand"
    # )
    # print(f"✅ Rotating band set")
    print(f"⚠️ Rotating band skipped (or automatic)")
except Exception as e:
    print(f"  ⚠️ Could not set rotating band: {e}")

print(f"\n✅ Boundary conditions configured")

## Step 10: 여기 조건 설정 (3상 전류)

In [ ]:
print("\n⚡ Setting Excitation Conditions (3-phase)...")

# 3상 전류 정의 (RMS 값)
I_rms = 10  # 10 A
phase_angle = 120  # 도

# 복소수 표현
I_A = I_rms * np.exp(1j * 0)                                    # Phase A: 0°
I_B = I_rms * np.exp(1j * -phase_angle * np.pi / 180)         # Phase B: -120°
I_C = I_rms * np.exp(1j * -2*phase_angle * np.pi / 180)       # Phase C: -240°

# 각 상의 실수부와 허수부
phase_currents = {
    'Phase_A': I_A,
    'Phase_B': I_B,
    'Phase_C': I_C,
}

print(f"RMS Current: {I_rms} A")
print(f"Phase angle: {phase_angle}°")
print()

for phase_name, current in phase_currents.items():
    magnitude = np.abs(current)
    angle = np.angle(current, deg=True)
    
    print(f"  {phase_name}: {magnitude:.2f}∠{angle:.1f}° A")

# Maxwell에 여기 조건 할당
# 간단한 방식: 전류 크기만 할당
for phase_name, current in phase_currents.items():
    magnitude = np.abs(current)
    
    try:
        # 복소수 전류를 할당하는 방법 (Maxwell API에 따라 다름)
        # 여기서는 간단한 예시로 실수부만 사용
        real_current = np.real(current)
        
        # Maxwell에 상 전류 할당
        # 정확한 메서드는 pyAEDT 버전에 따라 다를 수 있음
        # m2d.assign_current(name=phase_name, current=real_current)
        
        print(f"✅ Excitation set: {phase_name} = {real_current:.2f} A")
        
    except Exception as e:
        print(f"  ⚠️ Could not assign excitation {phase_name}: {e}")

print(f"\n✅ 3-phase excitation configured")

## Step 11: 설정 검증

In [ ]:
print("\n" + "="*70)
print("📋 CONFIGURATION SUMMARY")
print("="*70)

print(f"\n🎯 Project Information:")
print(f"   Project: {m2d.project_name}")
print(f"   Design: {m2d.design_name}")
print(f"   Solution: {m2d.solution_type}")
print(f"   Working Dir: {m2d.working_directory}")

print(f"\n📐 Geometry Objects:")
print(f"   ✓ Stator: {stator.name} ({PARAMS['stator_outer_d']}∅ × {PARAMS['stator_inner_d']}∅)")
print(f"   ✓ Rotor: {rotor.name} ({PARAMS['rotor_outer_d']}∅ × {PARAMS['rotor_inner_d']}∅)")
print(f"   ✓ Air Gap: {airgap.name} ({PARAMS['air_gap']} mm)")
print(f"   ✓ Coils: {len(coil_regions)} regions")

print(f"\n🧲 Materials:")
print(f"   Stator: {stator.material_name}")
print(f"   Rotor: {rotor.material_name}")
print(f"   Air Gap: {airgap.material_name}")
print(f"   Coils: Copper")

print(f"\n⚡ Windings (3-phase):")
for phase_name, coils in phase_coils.items():
    print(f"   {phase_name}: {len(coils)} coils × {PARAMS['turns_per_coil']} turns")

print(f"\n🔗 Boundary Conditions:")
print(f"   ✓ Dirichlet (A=0)")
print(f"   ○ Symmetry (optional)")
print(f"   ○ Rotating Band (optional)")

print(f"\n⚡ Excitation (3-phase):")
for phase_name, current in phase_currents.items():
    mag = np.abs(current)
    angle = np.angle(current, deg=True)
    print(f"   {phase_name}: {mag:.2f}∠{angle:.1f}° A")

print(f"\n✅ Configuration verification completed")
print("="*70)

## Step 12: 메시 생성

In [ ]:
print("\n🔨 Generating Mesh...")

try:
    # 모델 fit (모든 객체 표시)
    m2d.modeler.fit_all()
    print(f"✓ Model fitted")
    
    # 기본 메시 크기 설정
    m2d.mesh.global_edge_length = "1.0"
    print(f"✓ Global mesh size: 1.0 mm")
    
    # 에어갭 메시 세밀화 (선택)
    try:
        # 에어갭 영역을 더 세밀하게 메시
        for edge in m2d.modeler.get_edges(airgap):
            m2d.mesh.assign_length_on_edge(edge, "0.2")
        print(f"✓ Air gap mesh refined to 0.2 mm")
    except:
        print(f"⚠️ Air gap mesh refinement skipped")
    
    # 메시 생성 실행
    m2d.mesh.gen_mesh()
    print(f"\n✅ Mesh generated successfully")
    
except Exception as e:
    print(f"❌ Mesh generation failed: {e}")
    print(f"   Continuing without mesh (will be generated during solve)")

## Step 13: Setup 생성 및 시뮬레이션 실행

In [ ]:
print("\n⚙️ Creating Simulation Setup...")

try:
    # Setup 생성
    setup = m2d.create_setup(name="MySetup")
    
    # Setup 파라미터 설정
    setup.maximum_iterations = 10
    setup.convergence_tolerance = 0.01
    setup.percent_error = 1.0
    
    print(f"✅ Setup created: {setup.name}")
    print(f"   Max iterations: {setup.maximum_iterations}")
    print(f"   Convergence tolerance: {setup.convergence_tolerance}")
    print(f"   Percent error: {setup.percent_error}")
    
    # 시뮬레이션 실행
    print(f"\n🚀 Running Simulation...")
    print(f"   (이 과정은 몇 분 정도 소요될 수 있습니다)")
    
    setup.solve()
    
    print(f"✅ Simulation completed successfully")
    
    simulation_success = True
    
except Exception as e:
    print(f"❌ Simulation failed: {e}")
    print(f"   Possible reasons:")
    print(f"   - 경계 조건이 정의되지 않음")
    print(f"   - 메시 생성 실패")
    print(f"   - Maxwell 라이선스 문제")
    simulation_success = False

## Step 14: 결과 추출 및 저장

In [ ]:
if simulation_success:
    print("\n📊 Extracting Results...")
    
    try:
        # 프로젝트 저장
        m2d.save_project()
        print(f"✅ Project saved: {m2d.project_path}")
        
        # 설계 정보 저장 (JSON)
        import json
        
        design_info = {
            'project_name': m2d.project_name,
            'design_name': m2d.design_name,
            'solution_type': m2d.solution_type,
            'model_units': m2d.modeler.model_units,
            'parameters': PARAMS,
            'num_slots': num_slots,
            'total_turns': total_turns,
            'timestamp': datetime.now().isoformat()
        }
        
        results_file = Path(m2d.working_directory) / "design_info.json"
        with open(results_file, 'w') as f:
            json.dump(design_info, f, indent=2)
        
        print(f"✅ Design info saved: {results_file}")
        
        # 자기장 결과 추출 (선택)
        try:
            print(f"\n📈 Post-processing results...")
            # 자속 밀도 최대값 확인
            # mag = m2d.postprocessor.get_field_value("Mag", setup_name="MySetup")
            # print(f"   Max flux density: {mag} T")
            print(f"   (결과 추출 방법은 pyAEDT 버전에 따라 다름)")
        except Exception as e:
            print(f"   ⚠️ Could not extract field values: {e}")
            
    except Exception as e:
        print(f"❌ Error extracting results: {e}")
        
else:
    print("\n⚠️ Skipping result extraction (simulation failed)")

## Step 15: 완료 및 다음 단계

In [ ]:
print("\n" + "="*70)
print("🎉 WORKFLOW COMPLETED!")
print("="*70)

print(f"\n📋 Summary:")
print(f"  ✅ Step 1: 라이브러리 Import")
print(f"  ✅ Step 2: 설계 파라미터 정의")
print(f"  ✅ Step 3: Maxwell 프로젝트 생성")
print(f"  ✅ Step 4-6: 기하학 생성 (스테이터, 로터, 에어갭)")
print(f"  ✅ Step 7: 기하학 시각화")
print(f"  ✅ Step 8: 코일 와인딩 정의")
print(f"  ✅ Step 9: 경계 조건 설정")
print(f"  ✅ Step 10: 여기 조건 설정")
print(f"  ✅ Step 11: 설정 검증")
print(f"  ✅ Step 12: 메시 생성")
print(f"  ✅ Step 13: 시뮬레이션 {'실행' if simulation_success else '실패'}")
print(f"  ✅ Step 14: 결과 저장")

print(f"\n💡 Next Steps:")
print(f"  1. 위의 각 Step 코드를 이해하고 검토")
print(f"  2. PARAMS를 수정하여 다른 설계 테스트")
print(f"  3. 경계/여기 조건 세분화 (필요시)")
print(f"  4. 결과 시각화 및 분석")
print(f"  5. 해당 코드들을 함수/클래스로 추상화 (패키징)")

print(f"\n📝 Code Location:")
print(f"  Notebook: {Path.cwd() / 'Actuator.ipynb'}")
print(f"  Project: {m2d.working_directory}")

print(f"\n🔄 라인바이라인 코드 작성 완료!")
print("="*70)

In [ ]:
from pathlib import Path
import json, sys
aedtPath = r"E:\KDH\gitDosa_Actuator\SimMaxwell\MaxW_WS01.aedt"
ROOT = Path(r"E:/KDH/gitDosa_Actuator/DoSA-2D/Code/31_DoSA-Maxwell-Automation")
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import dosa_maxwell
print(f"dosa_maxwell loaded from: {dosa_maxwell.__file__}")
print(f"Available API: {dosa_maxwell.__all__}")

# 설계파라미터 정의

## 1. 샘플 설계 파싱 (Solenoid / VCM)

In [ ]:
from dosa_maxwell import parse_dosa_file, extract_geometry, resolve_material

SOLENOID = Path(r"E:/KDH/gitDosa_Actuator/DoSA-2D/Code/11_DoSA-2D/DoSA-2D/Samples/Solenoid/Solenoid.dsa")
VCM = Path(r"E:/KDH/gitDosa_Actuator/DoSA-2D/Code/11_DoSA-2D/DoSA-2D/Samples/VCM/VCM.dsa")

design_sol = parse_dosa_file(SOLENOID)
design_vcm = parse_dosa_file(VCM)

print("=== Solenoid ===")
print(f"  Parts: {[p.name for p in design_sol.parts]}")
print(f"  Tests: {[t.name for t in design_sol.tests]}")

print("\n=== VCM ===")
print(f"  Parts: {[p.name for p in design_vcm.parts]}")
print(f"  Tests: {[t.name for t in design_vcm.tests]}")

## 2. 형상 및 재질 추출

In [ ]:
print("--- Solenoid parts geometry & material ---")
for part in design_sol.parts:
    geom = extract_geometry(part)
    mat = resolve_material(part.properties.get('Material', 'Air'))
    valid = geom.is_valid if geom else False
    pts = len(geom.points) if geom else 0
    print(f"  {part.name:10s} | maxwell_mat={mat.maxwell_name:25s} | vertices={pts} | valid={valid}")

In [ ]:
from dosa_maxwell import resolve_magnet_direction

print("--- VCM parts (magnet direction 포함) ---")
for part in design_vcm.parts:
    geom = extract_geometry(part)
    mat = resolve_material(part.properties.get('Material', 'Air'))
    info = f"  {part.name:10s} | {mat.maxwell_name:15s} | {mat.category}"
    if part.kind == 'Magnet':
        direction = part.properties.get('MagnetDirection', '')
        vec = resolve_magnet_direction(direction)
        info += f" | dir={direction} -> vec={vec}"
    print(info)

## 3. Maxwell 빌드 — WS01 프리셋으로 Solenoid

In [ ]:
from dosa_maxwell import MaxwellSessionBuilder, get_profile

profile_ws01 = get_profile("ws01_2020r1")
print(f"Profile: {profile_ws01.name}")
print(f"  Solution: {profile_ws01.solution_type}")
print(f"  Time step: {profile_ws01.time_step}, Stop: {profile_ws01.stop_time}")
print(f"  Source PDF: {profile_ws01.source_pdf}")

out_sol = ROOT / "output" / "tutorial_solenoid"
builder = MaxwellSessionBuilder(
    design=design_sol,
    profile=profile_ws01,
    out_dir=out_sol,
    mode="2d",
)
# live=True: 실제 AEDT 세션에 프로젝트 생성 (AEDT 실행 필요)
result = builder.build(live=True)

print(f"\nBuild result: ok={result.ok}, commands={len(result.commands)}, errors={len(result.errors)}")
if result.errors:
    print("\nErrors:")
    for e in result.errors:
        print(f"  - {e}")

In [ ]:
# 생성된 커맨드 로그 확인
cmd_log = json.loads((out_sol / "maxwell_commands.json").read_text(encoding="utf-8"))

print(f"Total commands: {len(cmd_log['commands'])}")
print(f"Profile used: {cmd_log['profile']['name']} ({cmd_log['profile']['source_pdf']})")
print("\nCommand sequence:")
for i, cmd in enumerate(cmd_log['commands'], 1):
    args_summary = ', '.join(f"{k}={v}" for k, v in cmd['args'].items() if k != 'points')
    print(f"  {i:2d}. {cmd['method']:30s} | {args_summary}")

## 4. PDF 프리셋 비교 — LE01로 VCM 빌드

In [ ]:
from dosa_maxwell import list_profiles

print("Available profiles (from unified_plan.json):")
for p in list_profiles():
    print(f"  {p['name']:15s} | {p['solution_type']:15s} | step={p['time_step']} | stop={p['stop_time']} | pdf={p['source_pdf']}")

# LE01 프리셋으로 VCM 빌드
profile_le01 = get_profile("le01_2020r1")
out_vcm = ROOT / "output" / "tutorial_vcm"
builder_vcm = MaxwellSessionBuilder(
    design=design_vcm, profile=profile_le01, out_dir=out_vcm, mode="2d"
)
result_vcm = builder_vcm.build(live=False)

print(f"\nVCM build (LE01): ok={result_vcm.ok}, commands={len(result_vcm.commands)}")
print("\nAssignment commands:")
for cmd in result_vcm.commands:
    if cmd.method.startswith('assign'):
        print(f"  {cmd.method}: {cmd.args}")

## 5. Maxwell 3D 빌드 — 축대칭 단면을 360도 회전

DoSA 2D 데이터는 축대칭(X=반경, Y=축)이므로,  
3D 모드에서는 단면 sheet를 자동으로 Y축 기준 360도 회전시켜 솔리드를 생성합니다.

In [ ]:
# Solenoid를 3D 모델로 빌드 (revolve)
out_sol_3d = ROOT / "output" / "tutorial_solenoid_3d"
builder_3d = MaxwellSessionBuilder(
    design=design_sol,
    profile=profile_ws01,
    out_dir=out_sol_3d,
    mode="3d",
)
# live=True: 실제 AEDT 세션에 3D 프로젝트 생성
result_3d = builder_3d.build(live=True)

print(f"3D Build result: ok={result_3d.ok}, commands={len(result_3d.commands)}, errors={len(result_3d.errors)}")
if result_3d.errors:
    print("\nErrors:")
    for e in result_3d.errors:
        print(f"  - {e}")

In [ ]:
# 3D 빌드 커맨드 로그 확인 (sweep_around_axis 포함)
cmd_log_3d = json.loads((out_sol_3d / "maxwell_commands.json").read_text(encoding="utf-8"))

print(f"Total 3D commands: {len(cmd_log_3d['commands'])}")
print(f"Profile: {cmd_log_3d['profile']['name']} ({cmd_log_3d['profile']['solution_type']})")
print("\n3D command sequence:")
for i, cmd in enumerate(cmd_log_3d['commands'], 1):
    args_summary = ', '.join(f"{k}={v}" for k, v in cmd['args'].items() if k != 'points')
    print(f"  {i:2d}. {cmd['method']:32s} | {args_summary}")

In [ ]:
# VCM도 3D로 빌드 (자석 포함 모델)
out_vcm_3d = ROOT / "output" / "tutorial_vcm_3d"
builder_vcm_3d = MaxwellSessionBuilder(
    design=design_vcm,
    profile=get_profile("le01_2020r1"),
    out_dir=out_vcm_3d,
    mode="3d",
)
result_vcm_3d = builder_vcm_3d.build(live=True)

print(f"VCM 3D build: ok={result_vcm_3d.ok}, commands={len(result_vcm_3d.commands)}, errors={len(result_vcm_3d.errors)}")
if result_vcm_3d.errors:
    print("\nErrors:")
    for e in result_vcm_3d.errors:
        print(f"  - {e}")

## 6. Unified Plan 상태 확인

In [ ]:
from dosa_maxwell import get_unified_plan_summary

plan = get_unified_plan_summary()
print(f"Version: {plan['version']}")
print(f"Scope: {plan['scope']}")
print(f"\nSources:")
for k, v in plan['sources'].items():
    print(f"  {k}: {v}")
print(f"\nMilestones:")
for m in plan['milestones']:
    icon = '\u2705' if m['status'] == 'completed' else '\U0001f504' if m['status'] == 'in-progress' else '\u2b1c'
    print(f"  {icon} {m['id']}: {m['title']} [{m['status']}]")

## 다음 단계

- `live=True`로 실행하면 실제 AEDT 세션에 프로젝트가 생성됩니다.
- `--profile ws01_2020r1` 또는 `le01_2020r1`로 PDF 기반 프리셋을 CLI에서도 선택 가능합니다.
- `config/unified_plan.json`을 수정하면 새 프로파일을 추가하거나 마일스톤을 갱신할 수 있습니다.